In [ ]:
# =============================================================================
# FINAL 95% CONFIDENCE INTERVAL CALCULATION
# H=10 BINARY FUTURE WELD-STATE PREDICTION
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import t

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

N_RUNS = 10
CONFIDENCE_LEVEL = 0.95

# Degrees of freedom
df = N_RUNS - 1

# Two-sided Student's t critical value
alpha = 1.0 - CONFIDENCE_LEVEL
t_critical = t.ppf(1.0 - alpha / 2.0, df)

print("=" * 90)
print("FINAL 95% CI CALCULATION — STUDENT'S t-DISTRIBUTION")
print("=" * 90)

print(f"\nNumber of independent runs : {N_RUNS}")
print(f"Degrees of freedom         : {df}")
print(f"Confidence level           : {CONFIDENCE_LEVEL * 100:.0f}%")
print(f"Critical t-value           : {t_critical:.4f}")


# -----------------------------------------------------------------------------
# Final H=10 binary results
# Values are mean ± sample standard deviation across 10 independent runs
# -----------------------------------------------------------------------------

results = {
    "Accuracy": {
        "mean": 88.1558,
        "sd": 3.7666,
    },

    "Macro Precision": {
        "mean": 81.41,
        "sd": 4.43,
    },

    "Macro Recall": {
        "mean": 88.25,
        "sd": 5.11,
    },

    "Macro F1": {
        "mean": 83.77,
        "sd": 4.85,
    },

    "Weighted F1": {
        "mean": 88.75,
        "sd": 3.49,
    },
}


# -----------------------------------------------------------------------------
# Calculate Student's t-based 95% confidence intervals
# -----------------------------------------------------------------------------

rows = []

for metric, values in results.items():

    mean = values["mean"]
    sd = values["sd"]

    standard_error = sd / np.sqrt(N_RUNS)

    margin_of_error = (
        t_critical * standard_error
    )

    ci_lower = mean - margin_of_error
    ci_upper = mean + margin_of_error

    rows.append(
        {
            "Metric": metric,
            "Mean (%)": mean,
            "SD (%)": sd,
            "SE (%)": standard_error,
            "t critical": t_critical,
            "95% CI lower (%)": ci_lower,
            "95% CI upper (%)": ci_upper,
        }
    )


ci_df = pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# Display results
# -----------------------------------------------------------------------------

print("\n" + "=" * 90)
print("CORRECTED 95% CONFIDENCE INTERVALS")
print("=" * 90)

for _, row in ci_df.iterrows():

    print(
        f"\n{row['Metric']}:"
        f"\n  Mean ± SD : "
        f"{row['Mean (%)']:.2f} ± {row['SD (%)']:.2f}%"
        f"\n  SE        : "
        f"{row['SE (%)']:.4f}%"
        f"\n  95% CI    : "
        f"[{row['95% CI lower (%)']:.2f}, "
        f"{row['95% CI upper (%)']:.2f}]%"
    )


print("\n" + "=" * 90)
print("MANUSCRIPT TABLE")
print("=" * 90)

paper_df = ci_df[
    [
        "Metric",
        "Mean (%)",
        "SD (%)",
        "95% CI lower (%)",
        "95% CI upper (%)",
    ]
].copy()

paper_df = paper_df.round(2)

print(
    paper_df.to_string(
        index=False
    )
)


# -----------------------------------------------------------------------------
# Manuscript-ready text
# -----------------------------------------------------------------------------

print("\n" + "=" * 90)
print("MANUSCRIPT-READY VALUES")
print("=" * 90)

for _, row in ci_df.iterrows():

    print(
        f"{row['Metric']}: "
        f"{row['Mean (%)']:.2f} ± "
        f"{row['SD (%)']:.2f}% "
        f"(95% CI: "
        f"{row['95% CI lower (%)']:.2f}–"
        f"{row['95% CI upper (%)']:.2f}%)"
    )


# -----------------------------------------------------------------------------
# Verification of formula
# -----------------------------------------------------------------------------

print("\n" + "=" * 90)
print("FORMULA USED")
print("=" * 90)

print(
    "\n95% CI = mean ± "
    "t_(0.975, n-1) × SD / sqrt(n)"
)

print(
    f"\nFor n = {N_RUNS}:"
    f"\n  df = {df}"
    f"\n  t_(0.975, 9) = {t_critical:.4f}"
)

print("\nDone.")